# DiffusionSR vs Stochastic InterpolantCompares two runs that differ in **one** thing: the generative process.| | task 0 | task 2 ||---|---|---|| config | `baseline_ddpm_matched_temp` | `si_opt_a_sigma010_enc_temp` || `si_process` | `ddpm` | `interpolant` || encoder | `ablation_20260901/enc_temp` | *same* || conditioning | implicit, 64-ch | *same* || harness | EMA + grad clip + StepLR + FlippedDataset | *same* || sampling budget | DDIM skip=50 (20 steps) | 20 Heun steps |Everything else is byte-identical in the two configs.`task 11` (`baseline_diffusionsr_temp`, the as-published DiffusionSR trained by the plain`DiffusionModel.train()` loop) is loaded when present as a reference point, but it differs byfour harness features as well as by the process and is **not** the comparison.Reads the `results.json` / `predictions.npz` written by `slurm/interpolant/02_eval_array.sh`.

In [ ]:
# -- Configuration --------------------------------------------------------------import os, sysfrom pathlib import Path# TRACE paths; adjust if running elsewhere.REPO      = Path('/trace/group/forgelab/ngng/multifield/DiffusionSR_shohom')RUNS_ROOT = REPO / 'diffusionsr' / 'runs' / 'interpolant'EVAL_ROOT = REPO / 'eval_results' / 'interpolant'DATA_ROOT = Path('/trace/group/forgelab/ngng/multifield/data_fields')WANDB_PROJECT = 'Flow3D_SuperResolution'WANDB_ENTITY  = os.getenv('WANDB_ENTITY', '')# Melt-pool boundary threshold in Kelvin, matching analysis/evaluate_coupled.py.MP_THRESHOLD_K = 1900.0if str(REPO) not in sys.path:    sys.path.insert(0, str(REPO))

In [ ]:
# -- Imports --------------------------------------------------------------------import json, warningsimport numpy as npimport matplotlibmatplotlib.use('module://matplotlib_inline.backend_inline')import matplotlib.pyplot as pltimport matplotlib.image as mpimgimport pandas as pdwarnings.filterwarnings('ignore')plt.rcParams.update({'figure.dpi': 110, 'font.size': 9})

In [ ]:
# -- Run manifest ---------------------------------------------------------------# PRIMARY is the attributable comparison and must be present. OPTIONAL arms are loaded only if# their eval has run, so this notebook works after ARMS=0,2 and also after the full grid.PRIMARY = {    'DiffusionSR (matched)':   'si_baseline_ddpm_matched_temp',    'Interpolant A (s=0.10)':  'si_si_opt_a_sigma010_enc_temp',}OPTIONAL = {    'Interpolant A (s=0.05)':  'si_si_opt_a_sigma005_enc_temp',    'Interpolant A (s=0.20)':  'si_si_opt_a_sigma020_enc_temp',    'Interpolant B (c=0.10)':  'si_si_opt_b_c010_enc_temp',    'Interpolant B (c=0.25)':  'si_si_opt_b_c025_enc_temp',    'DiffusionSR (as-published)': 'si_baseline_diffusionsr_temp',}arms = {}for label, run in dict(**PRIMARY, **OPTIONAL).items():    eval_dir = EVAL_ROOT / run    results  = eval_dir / (run + '_results.json')    is_primary = label in PRIMARY    if results.exists():        arms[label] = {'run': run, 'eval_dir': eval_dir, 'results': results,                       'run_dir': RUNS_ROOT / run, 'primary': is_primary}    elif is_primary:        print('MISSING (required): ' + str(results))        print('   -> run:  ARMS=0,2 bash slurm/interpolant/submit_all.sh')print(str(len(arms)) + ' arm(s) loaded:')for label, a in arms.items():    tag = 'PRIMARY' if a['primary'] else 'optional'    print('  [' + tag.rjust(8) + '] ' + label.ljust(28) + a['run'])ARM_ORDER = [l for l in dict(**PRIMARY, **OPTIONAL) if l in arms]

## 1 - Metrics tableEvery number comes from the *same* metric code: `evaluate()` in `analysis/evaluate_coupled.py`.The baseline reaches it via `analysis/evaluate_baseline_matched.py`, which imports that functionand only swaps the model it builds -- so neither arm is scored by a different implementation.

In [ ]:
# -- Metrics table --------------------------------------------------------------rows = {}for label in ARM_ORDER:    with open(arms[label]['results']) as f:        rows[label] = json.load(f)metrics = pd.DataFrame(rows)# Lower is better for the error metrics; these are the exceptions.HIGHER_IS_BETTER = ('calibration', 'coverage', 'frac')def _is_higher_better(metric_name):    return any(k in str(metric_name).lower() for k in HIGHER_IS_BETTER)def _bold_best(df):    styled = df.style.format('{:.4g}')    for metric in df.index:        chooser = max if _is_higher_better(metric) else min        try:            best_col = chooser(df.columns, key=lambda c: float(df.loc[metric, c]))        except (TypeError, ValueError):            continue  # non-numeric row, e.g. a stored field name        styled = styled.apply(            lambda s, m=metric, b=best_col: [                'font-weight: bold; color: #0a7' if (s.name == m and col == b) else ''                for col in s.index],            axis=1)    return styledprint('Bold = best per row. Calibration is better NEAR ~0.95, not simply larger -- read it directly.')display(_bold_best(metrics))

In [ ]:
# -- Primary comparison: interpolant relative to the matched baseline -----------base_label, si_label = ARM_ORDER[0], ARM_ORDER[1]delta = pd.DataFrame({base_label: metrics[base_label], si_label: metrics[si_label]})with np.errstate(divide='ignore', invalid='ignore'):    denom = metrics[base_label].replace(0, np.nan)    delta['change %'] = 100.0 * (metrics[si_label] - metrics[base_label]) / denomdef _verdict(row):    if not np.isfinite(row['change %']):        return ''    improved = row['change %'] > 0 if _is_higher_better(row.name) else row['change %'] < 0    return 'better' if improved else 'worse'delta['interpolant is'] = delta.apply(_verdict, axis=1)display(delta.style.format({base_label: '{:.4g}', si_label: '{:.4g}', 'change %': '{:+.1f}%'}))

## 2 - Training curves and wall-clock time

In [ ]:
# -- Loss curves, read from the per-run .txt files (no W&B API call needed) -----fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))for label in ARM_ORDER:    run_dir = arms[label]['run_dir']    is_primary = arms[label]['primary']    for ax, fname, title in [(axes[0], 'loss_epoch.txt', 'train loss'),                             (axes[1], 'validation_loss_epoch.txt', 'val loss')]:        path = run_dir / fname        if path.exists():            curve = np.loadtxt(path).reshape(-1)            ax.plot(curve, '-' if is_primary else '--',                    lw=1.6 if is_primary else 1.0, label=label)        ax.set_title(title)        ax.set_xlabel('epoch')        ax.set_yscale('log')axes[0].legend(fontsize=7, frameon=False)plt.tight_layout()plt.show()print('NOTE: the loss VALUES are not comparable between arms. DDPM regresses onto epsilon;')print('the interpolant regresses onto the velocity x1 - x0. Use these to check convergence')print('behaviour within an arm, and the metrics table above to compare quality across arms.')

In [ ]:
# -- Wall-clock training time (both loops now write training_time.txt) ---------times = {}for label in ARM_ORDER:    path = arms[label]['run_dir'] / 'training_time.txt'    if path.exists():        times[label] = float(np.loadtxt(path)) / 3600.0if times:    fig, ax = plt.subplots(figsize=(6, 0.5 + 0.45 * len(times)))    colours = ['#0a7' if arms[l]['primary'] else '#bbb' for l in times]    ax.barh(list(times.keys()), list(times.values()), color=colours)    ax.set_xlabel('training time [h]')    ax.invert_yaxis()    for i, v in enumerate(times.values()):        ax.text(v, i, ' {:.1f}h'.format(v), va='center', fontsize=8)    plt.tight_layout()    plt.show()else:    print('No training_time.txt yet; it is written when a run completes.')

## 3 - Overlay plotsFirst the overlay PNG each evaluation rendered, then a rebuilt version from `predictions.npz`using the **same frame and the same colour scale** for every arm -- which the per-run PNGs donot guarantee, so only the rebuilt panels are safe to compare by eye.

In [ ]:
# -- The overlay PNGs written by each eval -------------------------------------pngs = []for label in ARM_ORDER:    for path in sorted(arms[label]['eval_dir'].glob('*overlay*.png')):        pngs.append((label, path))if pngs:    fig, axes = plt.subplots(len(pngs), 1, figsize=(11, 3.2 * len(pngs)))    for ax, (label, path) in zip(np.atleast_1d(axes), pngs):        ax.imshow(mpimg.imread(path))        ax.axis('off')        ax.set_title(label, fontsize=9)    plt.tight_layout()    plt.show()else:    print('No overlay PNGs found; the rebuilt panels below are the fallback.')

In [ ]:
# -- Matched frame, matched colour scale, rebuilt from predictions.npz ---------# predictions.npz holds pred (N, K, C, H, W) and gt (N, C, H, W) in PHYSICAL UNITS (Kelvin),# written with identical keys by both evaluators.def load_npz(label):    for cand in sorted(arms[label]['eval_dir'].glob('*.npz')):        return np.load(cand, allow_pickle=True)    return Noneloaded = {}for label in ARM_ORDER:    data = load_npz(label)    if data is not None:        loaded[label] = dataFRAME = -1   # last test frame: deep inside the quasi-steady windowCH = 0       # temperature channelif not loaded:    print('No predictions.npz found locally. Each eval attaches one to its W&B run; fetch with:')    print('  wandb artifact get <entity>/<project>/<run>_eval_predictions:latest')else:    gt = list(loaded.values())[0]['gt'][FRAME, CH]    vmin, vmax = 293.0, float(gt.max())   # ONE scale across all panels    fig, axes = plt.subplots(1, len(loaded) + 1, figsize=(3.0 * (len(loaded) + 1), 3.2))    axes[0].imshow(gt.T, origin='lower', cmap='jet', vmin=vmin, vmax=vmax)    axes[0].set_title('ground truth', fontsize=9)    axes[0].axis('off')    im = None    for ax, (label, data) in zip(axes[1:], loaded.items()):        pred = data['pred'][FRAME, :, CH].mean(axis=0)   # ensemble mean over K samples        im = ax.imshow(pred.T, origin='lower', cmap='jet', vmin=vmin, vmax=vmax)        mae = float(np.abs(pred - gt).mean())        ax.set_title(label + BS_N + 'ens-mean MAE {:.1f} K'.format(mae), fontsize=8)        ax.axis('off')    if im is not None:        fig.colorbar(im, ax=axes, fraction=0.02, label='T [K]')    plt.show()

## 4 - Melt-pool boundary contoursThe physically meaningful comparison: the 1900 K isotherm of each arm's ensemble mean, drawnagainst the ground truth.

In [ ]:
# -- Melt-pool contours --------------------------------------------------------PALETTE = ['#0a7', '#c33', '#36c', '#c80', '#90c', '#777']if loaded:    gt = list(loaded.values())[0]['gt'][FRAME, CH]    fig, ax = plt.subplots(figsize=(5.5, 4))    ax.imshow(gt.T, origin='lower', cmap='gray_r', vmin=293, vmax=float(gt.max()), alpha=0.35)    # Ground truth thick and black, so the arms are read against it.    ax.contour(gt.T, levels=[MP_THRESHOLD_K], colors='k', linewidths=2.4)    handles = [plt.Line2D([], [], color='k', lw=2.4, label='ground truth')]    for (label, data), colour in zip(loaded.items(), PALETTE):        pred = data['pred'][FRAME, :, CH].mean(axis=0)        style = '-' if arms[label]['primary'] else '--'        ax.contour(pred.T, levels=[MP_THRESHOLD_K], colors=[colour],                   linewidths=1.5, linestyles=style)        handles.append(plt.Line2D([], [], color=colour, lw=1.5, ls=style, label=label))    ax.legend(handles=handles, fontsize=7, frameon=False, loc='lower right')    ax.set_title('melt-pool boundary ({:.0f} K)'.format(MP_THRESHOLD_K), fontsize=9)    ax.axis('off')    plt.tight_layout()    plt.show()

## 5 - Ensemble spreadThe failure mode worth watching for. A tight data-dependent coupling can improve pointwiseerror *and* collapse sample diversity -- which would make the keyhole-variability metric worsewhile MAE looks better. Reproducing melt-pool variability is what the base paper exists for, soa narrower spread than the baseline is a regression, not a win.

In [ ]:
# -- Per-pixel ensemble std maps ----------------------------------------------have_std = {}for label, data in loaded.items():    if 'ens_std' in list(getattr(data, 'files', [])):        have_std[label] = dataif have_std:    smax = max(float(d['ens_std'][FRAME, CH].max()) for d in have_std.values())    fig, axes = plt.subplots(1, len(have_std), figsize=(3.2 * len(have_std), 3.2))    im = None    for ax, (label, data) in zip(np.atleast_1d(axes), have_std.items()):        spread = data['ens_std'][FRAME, CH]        im = ax.imshow(spread.T, origin='lower', cmap='magma', vmin=0, vmax=smax)  # shared scale        ax.set_title(label + BS_N + 'mean std {:.2f} K'.format(float(spread.mean())), fontsize=8)        ax.axis('off')    if im is not None:        fig.colorbar(im, ax=axes, fraction=0.02, label='ensemble std [K]')    plt.show()    print('Mean ensemble std over the whole test set (higher = more diverse samples):')    for label, data in have_std.items():        print('  ' + label.ljust(28) + '{:.3f} K'.format(float(data['ens_std'].mean())))    print()    print('If the interpolant is LOWER here but better on MAE, it has traded diversity for')    print('pointwise accuracy -- check the keyhole metrics in the table before calling it a win.')else:    print('No ens_std array in the npz files; re-run the eval to regenerate them.')

## 6 - Reading this correctly- **Tasks 0 vs 2 is the attributable comparison.** Their configs differ in `si_process` and  nothing else, so a difference here is the generative process and not the training harness.- **Check section 5 before declaring a win.** Better MAE with collapsed ensemble spread is a  regression against what DiffusionSR was built to do.- **`sigma` has no DDPM counterpart**, so it cannot be held equal. If the result turns out  sensitive to it, report the `sigma=0.05/0.20` arms (tasks 1 and 3) rather than one tuned point.- **One seed per arm** is not enough to separate a process effect from initialisation noise.  Treat the first pass as a signal check, not a result.